In [ ]:
from uuid import UUID

from dotenv import load_dotenv, find_dotenv
from db.conf import create_db_engine, get_async_session
from db.repositories.topics import TopicRepository
from db.repositories.videos import VideoRepository
from db.repositories.helpers import full_video_data
from core.agents.common import TemplateManager, gpt_5_nano, medium_effort_gpt_5
from core.agents.topic import TopicAgent, TopicName

load_dotenv(find_dotenv())
engine = create_db_engine()
db = get_async_session(engine)

agent = TopicAgent(
  gpt_5_nano(),
  medium_effort_gpt_5(),
  TemplateManager()
)

In [ ]:
async with db() as session:
  topic_repo = TopicRepository(session)
  all_topics = await topic_repo.get_all_topics()
  topics = [TopicName(id=t.id, name=t.name) for t in all_topics]

  video_repo = VideoRepository(session)
  video = await video_repo.get_video_by_id(UUID("142f935a-accb-11f0-b4c6-b7d5fd8c07bf"), with_meta=True, with_annotations=True)
  video_data = full_video_data(video)

video_data

In [ ]:
response = await agent.run(video_data, topics)
response